
# 1. Explore Data

In [56]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data") / "fiqa"

corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
qrels = pd.read_parquet(DATA_DIR / "qrels.parquet")

display(corpus.head())
display(queries.head())
display(qrels.head())

,_id,title,text
0,3,,I'm not saying I don't like the idea of on-the...
1,31,,So nothing preventing false ratings besides ad...
2,56,,You can never use a health FSA for individual ...
3,59,,Samsung created the LCD and other flat screen ...
4,63,,Here are the SEC requirements: The federal sec...


,_id,title,text
0,0,,What is considered a business expense on a bus...
1,4,,Business Expense - Car Insurance Deductible Fo...
2,5,,Starting a new online business
3,6,,“Business day” and “due date” for bills
4,7,,New business owner - How do taxes work for the...


,query-id,corpus-id,score
0,8,566392,1
1,8,65404,1
2,15,325273,1
3,18,88124,1
4,26,285255,1


In [57]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data") / "fiqa"  # Update this path if your data is located elsewhere
# DATA_DIR = Path("data").joinpath("fiqa") # Alternative way to construct the path  
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet") # read the corpus data from the parquet file into a pandas DataFrame 
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
qrels = pd.read_parquet(DATA_DIR / "qrels.parquet")

print(corpus.head())
display(queries.head())     
print(qrels.head())

  _id title                                               text
0   3        I'm not saying I don't like the idea of on-the...
1  31        So nothing preventing false ratings besides ad...
2  56        You can never use a health FSA for individual ...
3  59        Samsung created the LCD and other flat screen ...
4  63        Here are the SEC requirements: The federal sec...


,_id,title,text
0,0,,What is considered a business expense on a bus...
1,4,,Business Expense - Car Insurance Deductible Fo...
2,5,,Starting a new online business
3,6,,“Business day” and “due date” for bills
4,7,,New business owner - How do taxes work for the...


   query-id  corpus-id  score
0         8     566392      1
1         8      65404      1
2        15     325273      1
3        18      88124      1
4        26     285255      1


In [58]:

test_query_ids = set(qrels["query-id"].astype(str)) 
""" qrels["query-id"] selects the query-id column from the qrels DataFrame.
.astype(str) converts those IDs to strings.
set(...) makes a Python set of unique query IDs.

So test_query_ids becomes a set of all query IDs that have relevance labels. """    


test_queries = queries[queries["_id"].astype(str).isin(test_query_ids)]
""" queries["_id"] selects the _id column from the queries DataFrame.
.astype(str) converts query IDs to strings so they match test_query_ids.
.isin(test_query_ids) returns a boolean mask: True for rows whose _id is in the set.
queries[...] filters the queries DataFrame using that mask.

So test_queries contains only the query rows that have a matching relevance entry in qrels. """



print(f"Queries (test, with qrels): {len(test_queries)}")

Queries (test, with qrels): 648


### Inspect one query and its relevant docs

In [59]:
query = test_queries.iloc[0]
print("\nExample query")
print(f"  _id:  {query['_id']}")
print(f"  text: {query['text']}")

relevant = qrels[qrels["query-id"].astype(str) == str(query["_id"])]
print(f"\n  Relevant docs for this question: {list(relevant['corpus-id'])}")


Example query
  _id:  4641
  text: Where should I park my rainy-day / emergency fund?

  Relevant docs for this question: [44594, 406219, 319954, 397358, 88327]


# bm25.py  
### Step 1. Load the Corpus
 FiQA docs are forum posts; the 'title' column is empty for every row,
 so we index on 'text' directly

In [60]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
doc_ids = corpus["_id"].tolist()
doc_texts = corpus["text"].tolist()     

print(f"Indexing {len(doc_texts)} documents with BM25...")

Indexing 57638 documents with BM25...


### Step 2: Tokenize and build the index


bm25s.tokenize lowercases, strips punctuation, and removes English stopwords. 
The result is a Tokenized object that you hand straight to BM25.index().

In [61]:
tokens = bm25s.tokenize(doc_texts, stopwords="en")    

"""
What tokenize does (typical behavior): lowercases text, strips punctuation, splits into tokens, and removes stopwords (if requested).
Return value: a Tokenized object with at least:
tokens.ids — list[list[int]]: token-id sequence for each document (one inner list per doc).
"""

print(tokens.ids[:1])  # list[list[int]] -- one inner list per doc
"""
tokens.ids is a Python list where each element is a list of integer token IDs
 for one document: hence type [list[list[int]]](a list of lists of integers).
"""
print(list(tokens.vocab.items())[:10])  # dict[str, int] -- token string -> integer ID

"""
tokens.vocab — a Python dict mapping token string → integer id (type: dict[str, int]).
.items() — returns an iterable view of (token, id) pairs.  
 """

retriever = bm25s.BM25()  # method='lucene' by default  
retriever.index(tokens)   


""" 
retriever = bm25s.BM25()

Creates a BM25 retriever object. By default it uses the "lucene" parameterization (BM25 params tuned like Lucene: defaults for k1, b).
retriever.index(tokens)

tokens is the Tokenized object returned by bm25s.tokenize(...).
This call builds the internal BM25 index: computes term frequencies, document lengths, IDF scores, and any lookup tables needed for fast scoring. It does not return results — it prepares retriever to answer queries.
After indexing you can score/search queries (example):
 """

Split strings:   0%|          | 0/57638 [00:00<?, ?it/s]

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 5, 12, 4, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 24, 33, 34, 35, 36, 37, 38, 39, 11, 40]]
[('saying', 0), ('don', 1), ('like', 2), ('idea', 3), ('job', 4), ('training', 5), ('too', 6), ('you', 7), ('can', 8), ('expect', 9)]


BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/57638 [00:00<?, ?it/s]

' \nretriever = bm25s.BM25()\n\nCreates a BM25 retriever object. By default it uses the "lucene" parameterization (BM25 params tuned like Lucene: defaults for k1, b).\nretriever.index(tokens)\n\ntokens is the Tokenized object returned by bm25s.tokenize(...).\nThis call builds the internal BM25 index: computes term frequencies, document lengths, IDF scores, and any lookup tables needed for fast scoring. It does not return results — it prepares retriever to answer queries.\nAfter indexing you can score/search queries (example):\n '

### Step 3: Persist the index to disk  
Save the index plus the doc_ids in matching order, so we can map back later.

In [62]:
INDEX_DIR.mkdir(parents=True, exist_ok=True)
retriever.save(str(INDEX_DIR))
(INDEX_DIR / "doc_ids.txt").write_text("\n".join(doc_ids))   

""" 
INDEX_DIR.mkdir(parents=True, exist_ok=True)

Create the directory INDEX_DIR and any missing parent folders. exist_ok=True prevents an error if the folder already exists.
retriever.save(str(INDEX_DIR))

Persist the BM25 index files into that directory so you can reload the index later without re-tokenizing/re-indexing. The exact files/format are produced by bm25s.
(INDEX_DIR / "doc_ids.txt").write_text("\n".join(doc_ids))

Save the doc_ids list (one original document ID per line) in the same order the documents were indexed. This file lets you map BM25 result indices back to the original document IDs/text.
  """

' \nINDEX_DIR.mkdir(parents=True, exist_ok=True)\n\nCreate the directory INDEX_DIR and any missing parent folders. exist_ok=True prevents an error if the folder already exists.\nretriever.save(str(INDEX_DIR))\n\nPersist the BM25 index files into that directory so you can reload the index later without re-tokenizing/re-indexing. The exact files/format are produced by bm25s.\n(INDEX_DIR / "doc_ids.txt").write_text("\n".join(doc_ids))\n\nSave the doc_ids list (one original document ID per line) in the same order the documents were indexed. This file lets you map BM25 result indices back to the original document IDs/text.\n  '

### Step 4 : Run a Query 

In [63]:
def search_bm25(query: str, k: int = 10) -> list[tuple[str, float]]:
    """Return the top-k (doc_id, score) pairs for a query."""
    query_tokens = bm25s.tokenize([query], stopwords="en")
    indices, scores = retriever.retrieve(query_tokens, k=k)
    # indices[0] is a numpy array of integer positions in doc_ids.
    return [
        (doc_ids[i], float(scores[0][j])) for j, i in enumerate(indices[0].tolist())
    ]       

In [64]:
if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"\nQuery: {query}\n")
    for i, (doc_id, score) in enumerate(search_bm25(query, k=5), 1):
        text = corpus.loc[corpus["_id"] == doc_id, "text"].iloc[0]
        print(f"{i}. [{score:6.2f}] {doc_id}  {text[:80]}")


Query: Where should I park my rainy-day fund?



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

1. [ 13.76] 376148  Bond aren't necessarily any safer than the stock market.  Ultimately, there is n
2. [  8.88] 32833  In addition to the issues discussed in BrenBarn's answer, I think you need to co
3. [  7.91] 416189  For me there are two issues.  So, what to do? You have the basics of a very stro
4. [  7.83] 579848  You are only 33, with plenty of time to generate long-term returns. A correction
5. [  7.59] 580025  "I don't know Canada very well, but can offer some general points when consideri


# 3. Embed 

In [65]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [66]:
#%pip install sentence-transformers 

In [67]:
# Initialize the open-source model (Runs completely free and locally)
# 'all-MiniLM-L6-v2' is fast and lightweight (384 dimensions vs OpenAI's 1536)
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

DATA_DIR = Path("data") / "fiqa"  
INDEX_DIR = Path("indexes") / "dense" 
INDEX_DIR.mkdir(parents=True, exist_ok=True)   
#DATA_DIR = Path(__file__).parent / "data" / "fiqa"
#INDEX_DIR = Path(__file__).parent / "indexes" / "dense"
#INDEX_DIR.mkdir(parents=True, exist_ok=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Step 1. Embed the corpus in batches 

In [68]:
def embed_batch(texts: list[str]) -> np.ndarray:
    """Embed a batch of texts and return a numpy array."""
    # sentence-transformers handles the processing internally and returns a numpy array
    return model.encode(texts, convert_to_numpy=True, show_progress_bar=False)


def build_index(doc_texts: list[str], batch_size: int = 256) -> np.ndarray:
    """Embed the full corpus in batches with a progress bar."""
    chunks = []
    for i in tqdm(range(0, len(doc_texts), batch_size), desc="Embedding"):
        chunks.append(embed_batch(doc_texts[i : i + batch_size]))
    return np.vstack(chunks)  # stack batches into one matrix

### Step 2. Build or load the cached embeddings matrix 

In [69]:
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
doc_ids = corpus["_id"].tolist()

# Handle blank documents safely
doc_texts = [t.strip() or "[empty document]" for t in corpus["text"].tolist()]

# We include the model name in the filename to prevent loading an OpenAI cache matrix
embeddings_path = INDEX_DIR / f"embeddings_{model_name}.npy"

if embeddings_path.exists():
    print(f"Loading cached embeddings from {embeddings_path}")
    doc_embeddings = np.load(embeddings_path)
else:
    print(f"Embedding {len(doc_texts)} docs locally using {model_name} ($0.00 cost)")
    doc_embeddings = build_index(doc_texts)
    np.save(embeddings_path, doc_embeddings)

# Pre-normalize once so cosine similarity becomes a single dot product later.
doc_embeddings_normed = doc_embeddings / np.linalg.norm(
    doc_embeddings, axis=1, keepdims=True
)

Loading cached embeddings from indexes\dense\embeddings_all-MiniLM-L6-v2.npy


### Step 3. Query by cosine similarity 

In [70]:
def search_dense(query: str, k: int = 10) -> list[tuple[str, float]]:
    """Return the top-k (doc_id, similarity) pairs for a query."""
    query_vec = embed_batch([query])[0]
    query_vec /= np.linalg.norm(query_vec)
    scores = doc_embeddings_normed @ query_vec
    top_k = np.argsort(-scores)[:k]
    return [(doc_ids[i], float(scores[i])) for i in top_k]


In [71]:

if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"\nQuery: {query}\n")
    for i, (doc_id, score) in enumerate(search_dense(query, k=5), 1):
        text = corpus.loc[corpus["_id"] == doc_id, "text"].iloc[0]
        print(f"{i}. [{score:.3f}] {doc_id} {text}\n")


Query: Where should I park my rainy-day fund?

1. [0.566] 553748 "It sounds like you want a place to park some money that's reasonably safe and liquid, but can sustain light to moderate losses. Consider some bond funds or bond ETFs filled with medium-term corporate bonds. It looks like you can get 3-3.5% or so. (I'd skip the municipal bond market right now, but ""why"" is a matter for its own question). Avoid long-term bonds or CDs if you're worried about inflation; interest rates will rise and the immediate value of the bonds will fall until the final payout value matches those rates."

2. [0.563] 425479 "As you're saving up for an expenditure instead of investing for the long run, I would stay away from any sort of ""parking facility"" where you run the risk of not having the principal protected. The riskier investments that would potentially generate a bigger return also carry a bigger downside, ie you might not be able to get the money back that you put in. I'd shop around for a C

# 4. RRF(reciprocal rank fusing) 

"""
Reciprocal Rank Fusion (RRF). Combine BM25 and dense retrieval into one
ranked list.

The naive idea is 'average the scores', but BM25 scores are unbounded and
cosine similarities sit in [0, 1]. The fix is to fuse RANKINGS, not scores.

    rrf_score(d) = sum over each retriever r of 1 / (k + rank_r(d))

k is a smoothing constant, conventionally 60. The original 2009 paper called
it 'simple but effective' and that is still the consensus in 2026.

More info: https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf
"""

In [72]:
import numpy as np
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# We drop DenseRetriever from the import, but keep BM25 and the corpus loader
from utils.retrievers import BM25Retriever, load_corpus

K_RRF = 60
INDEX_DIR = Path("indexes") / "dense"

#### Step 1. The fusion function 

In [73]:
def reciprocal_rank_fusion(
    rankings: list[list[str]], k: int = K_RRF
) -> list[tuple[str, float]]:
    """Fuse multiple ranked lists of doc_ids into one ranked list."""
    scores: dict[str, float] = defaultdict(float)
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

#### Step2. Define and load both retrievers 

In [74]:
class LocalDenseRetriever:
    """A custom retriever that uses the free, local sentence-transformers model."""
    def __init__(self, corpus_df):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_ids = corpus_df["_id"].tolist()
        
        # Load the locally generated embedding matrix from the previous step
        matrix_path = INDEX_DIR / "embeddings_all-MiniLM-L6-v2.npy"
        if not matrix_path.exists():
            raise FileNotFoundError(f"Missing {matrix_path}. Run the embedding script first!")
            
        embeddings = np.load(matrix_path)
        # Pre-normalize for fast cosine similarity dot products
        self.embeddings_normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_vec = self.model.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
        query_vec /= np.linalg.norm(query_vec)
        scores = self.embeddings_normed @ query_vec
        top_k = np.argsort(-scores)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k]


# Load corpus and initialize both retrievers
corpus = load_corpus()
bm25 = BM25Retriever()
dense = LocalDenseRetriever(corpus)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#### Step 3. Search both , fuse , compare

In [75]:
def search_hybrid(
    query: str, k: int = 10, candidate_k: int = 50
) -> list[tuple[str, float]]:
    """Retrieve top candidate_k from each retriever, fuse, return top k."""
    bm25_ids = [doc_id for doc_id, _ in bm25.search(query, k=candidate_k)]
    dense_ids = [doc_id for doc_id, _ in dense.search(query, k=candidate_k)]
    return reciprocal_rank_fusion([bm25_ids, dense_ids])[:k]


def show(label: str, results: list[tuple[str, float]]) -> None:
    print(f"\n{label}")
    for i, (doc_id, score) in enumerate(results[:5], 1):
        text = corpus.loc[corpus["_id"] == doc_id, "text"].iloc[0]
        print(f"  {i}. [{score:.4f}] {doc_id}  {text[:70]}")

In [76]:
if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"Query: {query}")

    show("BM25 only", bm25.search(query, k=5))
    show("Dense only", dense.search(query, k=5))
    show("Hybrid (RRF)", search_hybrid(query, k=5))

Query: Where should I park my rainy-day fund?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


BM25 only
  1. [13.7607] 376148  Bond aren't necessarily any safer than the stock market.  Ultimately, 
  2. [8.8848] 32833  In addition to the issues discussed in BrenBarn's answer, I think you 
  3. [7.9143] 416189  For me there are two issues.  So, what to do? You have the basics of a
  4. [7.8303] 579848  You are only 33, with plenty of time to generate long-term returns. A 
  5. [7.5942] 580025  "I don't know Canada very well, but can offer some general points when

Dense only
  1. [0.5658] 553748  "It sounds like you want a place to park some money that's reasonably 
  2. [0.5625] 425479  "As you're saving up for an expenditure instead of investing for the l
  3. [0.5605] 44594  "First off, you generally want to park your emergency fund somewhere t
  4. [0.5501] 406219  I would suggest your local credit union or local bank for security and
  5. [0.5430] 254572  From a budgeting perspective, the emergency fund is a category in whic


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Hybrid (RRF)
  1. [0.0306] 32833  In addition to the issues discussed in BrenBarn's answer, I think you 
  2. [0.0301] 376148  Bond aren't necessarily any safer than the stock market.  Ultimately, 
  3. [0.0292] 497993  Duffbeer703 covers most everything. The entire point of an emergency f
  4. [0.0264] 261983  A questoin that I deal with almost every day.  Like most investments i
  5. [0.0226] 386305  Thank you for your service.  My first suggestion since your car is a p


# 5. Re-Rank

"""
Reranking: take the top-50 candidates from RRF and reorder them with a
cross-encoder. Return the top-10.

A bi-encoder (the dense retriever from 3-embed.py) embeds the query and the
document separately, then compares with cosine. A cross-encoder feeds the
query and document INTO the same model and returns a single relevance score.
Cross-encoders are slower, but the joint attention catches subtleties that
two independent embeddings miss.

We use Cohere rerank-v4.0-fast.

More info: https://docs.cohere.com/docs/rerank-overview
"""

In [77]:
import os
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
  
# We import the clean BM25 retriever and the corpus loader
from utils.retrievers import BM25Retriever, load_corpus

# Setup paths matching your directory structure
# ROOT = Path(__file__).parent.parent  
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "fiqa"
DENSE_DIR = ROOT / "indexes" / "dense"

# Initialize the free, open-source cross-encoder reranker model
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL)

K_RRF = 60

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

#### Define local retrievers and fusion 

In [78]:
class LocalDenseRetriever:
    """A custom retriever that uses the free, local sentence-transformers model."""
    def __init__(self, corpus_df) -> None:
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_ids = corpus_df["_id"].tolist()
        
        # Load the locally generated embedding matrix from the previous step
        matrix_path = DENSE_DIR / "embeddings_all-MiniLM-L6-v2.npy"
        if not matrix_path.exists():
            raise FileNotFoundError(f"Missing {matrix_path}. Run your embedding script first!")
            
        raw = np.load(matrix_path)
        self._embeddings = raw / np.linalg.norm(raw, axis=1, keepdims=True)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_vec = self.model.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
        query_vec /= np.linalg.norm(query_vec)
        scores = self._embeddings @ query_vec
        top_k = np.argsort(-scores)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k]


def local_hybrid_candidates(query: str, bm25: BM25Retriever, dense: LocalDenseRetriever, candidate_k: int = 50) -> list[tuple[str, float]]:
    """Fuses BM25 and Dense local rankings using Reciprocal Rank Fusion (RRF)."""
    bm25_ids = [doc_id for doc_id, _ in bm25.search(query, k=candidate_k)]
    dense_ids = [doc_id for doc_id, _ in dense.search(query, k=candidate_k)]
    
    scores: dict[str, float] = defaultdict(float)
    for ranking in [bm25_ids, dense_ids]:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (K_RRF + rank)
            
    return sorted(scores.items(), key=lambda x: -x[1])

#### running embedding script error

In [81]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Initialize the open-source model
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

# Define the absolute root path targeting the hybrid-retrieval subdirectory
ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge\hybrid-retrieval")
DATA_DIR = ROOT / "data" / "fiqa"
INDEX_DIR = ROOT / "indexes" / "dense"

# Fallback path checking: If the subfolder path is wrong, fall back to the parent folder
if not (DATA_DIR / "corpus.parquet").exists():
    ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge")
    DATA_DIR = ROOT / "data" / "fiqa"
    INDEX_DIR = ROOT / "indexes" / "dense"

# Ensure directories exist
INDEX_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# Step 1: Embed the corpus in batches
# --------------------------------------------------------------

def embed_batch(texts: list[str]) -> np.ndarray:
    return model.encode(texts, convert_to_numpy=True, show_progress_bar=False)

def build_index(doc_texts: list[str], batch_size: int = 256) -> np.ndarray:
    chunks = []
    for i in tqdm(range(0, len(doc_texts), batch_size), desc="Embedding Corpus"):
        chunks.append(embed_batch(doc_texts[i : i + batch_size]))
    return np.vstack(chunks)

# --------------------------------------------------------------
# Step 2: Build or load the cached embedding matrix
# --------------------------------------------------------------

print(f"Loading corpus from {DATA_DIR / 'corpus.parquet'}...")
corpus = pd.read_parquet(DATA_DIR / "corpus.parquet")
doc_texts = [t.strip() or "[empty document]" for t in corpus["text"].tolist()]

embeddings_path = INDEX_DIR / f"embeddings_{model_name}.npy"

print(f"Generating vectors and saving to: {embeddings_path}")
doc_embeddings = build_index(doc_texts)
np.save(embeddings_path, doc_embeddings)
print("Embedding process complete! File saved successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading corpus from c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge\hybrid-retrieval\data\fiqa\corpus.parquet...
Generating vectors and saving to: c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge\hybrid-retrieval\indexes\dense\embeddings_all-MiniLM-L6-v2.npy


Embedding Corpus: 100%|██████████| 226/226 [10:44<00:00,  2.85s/it]

Embedding process complete! File saved successfully.


## ReRank Script


In [83]:
import os
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
from utils.retrievers import BM25Retriever, load_corpus

# Target the hybrid-retrieval folder structure
ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge\hybrid-retrieval")
DATA_DIR = ROOT / "data" / "fiqa"
DENSE_DIR = ROOT / "indexes" / "dense"

# Fallback checking to match the embedding script configuration
if not (DATA_DIR / "corpus.parquet").exists():
    ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge")
    DATA_DIR = ROOT / "data" / "fiqa"
    DENSE_DIR = ROOT / "indexes" / "dense"

# Initialize the free cross-encoder model
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL)
K_RRF = 60

# --------------------------------------------------------------
# Classes and Retrieval Logic
# --------------------------------------------------------------

class LocalDenseRetriever:
    def __init__(self, corpus_df) -> None:
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_ids = corpus_df["_id"].tolist()
        
        matrix_path = DENSE_DIR / "embeddings_all-MiniLM-L6-v2.npy"
        if not matrix_path.exists():
            raise FileNotFoundError(f"Missing {matrix_path}. Run your embedding script first!")
            
        raw = np.load(matrix_path)
        self._embeddings = raw / np.linalg.norm(raw, axis=1, keepdims=True)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_vec = self.model.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
        query_vec /= np.linalg.norm(query_vec)
        scores = self._embeddings @ query_vec
        top_k = np.argsort(-scores)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k]


def local_hybrid_candidates(query: str, bm25: BM25Retriever, dense: LocalDenseRetriever, candidate_k: int = 50) -> list[tuple[str, float]]:
    bm25_ids = [doc_id for doc_id, _ in bm25.search(query, k=candidate_k)]
    dense_ids = [doc_id for doc_id, _ in dense.search(query, k=candidate_k)]
    
    scores: dict[str, float] = defaultdict(float)
    for ranking in [bm25_ids, dense_ids]:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (K_RRF + rank)
            
    return sorted(scores.items(), key=lambda x: -x[1])


def search_reranked(query: str, k: int = 10, candidate_k: int = 50) -> list[tuple[str, float]]:
    candidates = local_hybrid_candidates(query, bm25, dense, candidate_k=candidate_k)
    candidate_ids = [doc_id for doc_id, _ in candidates]
    candidate_texts = [corpus_by_id.loc[d, "text"] for d in candidate_ids]

    pairs = [[query, text] for text in candidate_texts]
    relevance_scores = reranker.predict(pairs)

    reranked_results = list(zip(candidate_ids, relevance_scores))
    reranked_results.sort(key=lambda x: -x[1])
    return reranked_results[:k]


def show(label: str, results: list[tuple[str, float]]) -> None:
    print(f"\n{label}")
    for i, (doc_id, score) in enumerate(results[:5], 1):
        text = corpus_by_id.loc[doc_id, "text"]
        print(f"  {i}. [{score:.4f}] {doc_id}  {text[:70]}")


# --------------------------------------------------------------
# Execution Execution Flow
# --------------------------------------------------------------

corpus_df = pd.read_parquet(DATA_DIR / "corpus.parquet")
corpus_by_id = corpus_df.set_index("_id")

bm25 = BM25Retriever()
dense = LocalDenseRetriever(corpus_df)

if __name__ == "__main__":
    query = "Where should I park my rainy-day fund?"
    print(f"Query: {query}")

    show("Hybrid (RRF) only", local_hybrid_candidates(query, bm25, dense, candidate_k=50)[:5])
    show(f"Hybrid + Local Rerank ({RERANK_MODEL})", search_reranked(query, k=5))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: Where should I park my rainy-day fund?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Hybrid (RRF) only
  1. [0.0306] 32833  In addition to the issues discussed in BrenBarn's answer, I think you 
  2. [0.0301] 376148  Bond aren't necessarily any safer than the stock market.  Ultimately, 
  3. [0.0292] 497993  Duffbeer703 covers most everything. The entire point of an emergency f
  4. [0.0264] 261983  A questoin that I deal with almost every day.  Like most investments i
  5. [0.0226] 386305  Thank you for your service.  My first suggestion since your car is a p


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Hybrid + Local Rerank (cross-encoder/ms-marco-MiniLM-L-6-v2)
  1. [4.4663] 376148  Bond aren't necessarily any safer than the stock market.  Ultimately, 
  2. [1.8246] 580025  "I don't know Canada very well, but can offer some general points when
  3. [-1.4273] 8088  Only having 1200 dollars sounds extremely dangerous and in my opinion 
  4. [-2.0637] 32833  In addition to the issues discussed in BrenBarn's answer, I think you 
  5. [-2.0660] 253614  "Liquid cash (emergency, rainy day fund) should be safe from a loss in


# 6. Evaluate 

In [84]:
import os
import math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm  # Optimized progress bar for Jupyter Notebooks
from sentence_transformers import SentenceTransformer, CrossEncoder

# We keep the clean BM25 retriever and corpus loader from your utils module
from utils.retrievers import BM25Retriever, load_corpus

# ==============================================================================
# 1. ROBUST PATH RESOLUTION
# ==============================================================================
ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge\hybrid-retrieval")
DATA_DIR = ROOT / "data" / "fiqa"
DENSE_DIR = ROOT / "indexes" / "dense"

# Fallback pathing check to prevent FileNotFoundError if notebook context shifts
if not (DATA_DIR / "queries.parquet").exists():
    ROOT = Path(r"c:\Users\user2\Desktop\IITR_internship_i23ma003\Rag\ai-cookbook-main\knowledge")
    DATA_DIR = ROOT / "data" / "fiqa"
    DENSE_DIR = ROOT / "indexes" / "dense"

# ==============================================================================
# 2. LOCAL MODEL CONFIGURATIONS (FREE TIER ALTERNATIVES)
# ==============================================================================
RERANK_SAMPLE_SIZE = 50
SEED = 42

# Initialize local models directly in memory
print("Loading open-source embedding & cross-encoder models into memory...")
local_bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
local_cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
K_RRF = 60

# ==============================================================================
# 3. COMPACT LOCAL COMPONENTS
# ==============================================================================
class LocalDenseRetriever:
    """Retrieves documents locally using precalculated MiniLM matrix values."""
    def __init__(self, corpus_df) -> None:
        self.doc_ids = corpus_df["_id"].tolist()
        matrix_path = DENSE_DIR / "embeddings_all-MiniLM-L6-v2.npy"
        if not matrix_path.exists():
            raise FileNotFoundError(f"Missing matrix file: {matrix_path}. Build embeddings first!")
        raw = np.load(matrix_path)
        self._embeddings = raw / np.linalg.norm(raw, axis=1, keepdims=True)

    def search(self, query: str, k: int = 10) -> list[tuple[str, float]]:
        query_vec = local_bi_encoder.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
        query_vec /= np.linalg.norm(query_vec)
        scores = self._embeddings @ query_vec
        top_k = np.argsort(-scores)[:k]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k]

def local_hybrid_candidates(query: str, bm25_retriever, dense_retriever, candidate_k: int = 50) -> list[str]:
    """Combines BM25 and Dense search lists using Reciprocal Rank Fusion."""
    bm25_ids = [doc_id for doc_id, _ in bm25_retriever.search(query, k=candidate_k)]
    dense_ids = [doc_id for doc_id, _ in dense_retriever.search(query, k=candidate_k)]
    
    scores = defaultdict(float)
    for ranking in [bm25_ids, dense_ids]:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] += 1.0 / (K_RRF + rank)
            
    sorted_candidates = sorted(scores.items(), key=lambda x: -x[1])
    return [doc_id for doc_id, _ in sorted_candidates]

def local_rerank_topk(query: str, candidate_ids: list[str], corpus_by_id_df, k: int = 10) -> list[str]:
    """Reranks candidates using local cross-encoder transformer attention frames."""
    if not candidate_ids:
        return []
    candidate_texts = [corpus_by_id_df.loc[d, "text"] for d in candidate_ids]
    pairs = [[query, text] for text in candidate_texts]
    
    relevance_scores = local_cross_encoder.predict(pairs)
    reranked_results = list(zip(candidate_ids, relevance_scores))
    reranked_results.sort(key=lambda x: -x[1])
    return [doc_id for doc_id, _ in reranked_results[:k]]

def ndcg_at_k(predicted_ids: list[str], relevant_dict: dict[str, int], k: int = 10) -> float:
    """Calculates Normalized Discounted Cumulative Gain at position K."""
    dcg = sum(
        relevant_dict.get(doc_id, 0) / math.log2(rank + 2)
        for rank, doc_id in enumerate(predicted_ids[:k])
    )
    ideal_rels = sorted(relevant_dict.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(rank + 2) for rank, rel in enumerate(ideal_rels))
    return dcg / idcg if idcg > 0 else 0.0

# ==============================================================================
# 4. DATA LOADING AND SAMPLING
# ==============================================================================
print("Loading FiQA evaluation parquets...")
queries = pd.read_parquet(DATA_DIR / "queries.parquet")
qrels_df = pd.read_parquet(DATA_DIR / "qrels.parquet")
corpus_df = load_corpus()
corpus_by_id = corpus_df.set_index("_id")

# Process the evaluation lookup tables
qrels = defaultdict(dict)
for _, row in qrels_df.iterrows():
    qrels[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])

queries_with_qrels = queries[queries["_id"].astype(str).isin(qrels.keys())].copy()
sample = queries_with_qrels.sample(n=RERANK_SAMPLE_SIZE, random_state=SEED)
print(f"Evaluating across {len(sample)} sample items out of {len(queries_with_qrels)} validation queries.")

# Instantiate localized retrievers
bm25 = BM25Retriever()
dense = LocalDenseRetriever(corpus_df)

# ==============================================================================
# 5. RUN EVALUATION LOOP
# ==============================================================================
results = defaultdict(list)

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Evaluating RAG Pipelines"):
    query_id = str(row["_id"])
    query_text = row["text"]
    relevant = qrels[query_id]

    # Pipeline A: Standard Key-Word matching
    bm25_predictions = [d for d, _ in bm25.search(query_text, k=10)]
    results["BM25"].append(ndcg_at_k(bm25_predictions, relevant, k=10))

    # Pipeline B: Local Dense Vector Semantic matching
    dense_predictions = [d for d, _ in dense.search(query_text, k=10)]
    results["Dense (MiniLM)"].append(ndcg_at_k(dense_predictions, relevant, k=10))

    # Pipeline C: Balanced Hybrid fusion (RRF)
    hybrid_candidates_list = local_hybrid_candidates(query_text, bm25, dense, candidate_k=50)
    results["Hybrid (RRF)"].append(ndcg_at_k(hybrid_candidates_list[:10], relevant, k=10))

    # Pipeline D: Two-stage retrieval using local Cross-Encoder reranking
    reranked_predictions = local_rerank_topk(query_text, hybrid_candidates_list, corpus_by_id, k=10)
    results["Hybrid + Local Rerank"].append(ndcg_at_k(reranked_predictions, relevant, k=10))

# ==============================================================================
# 6. OUTPUT SUMMARY MATRIX
# ==============================================================================
print(f"\nNDCG@10 x100 Results Table ({len(sample)} sample items)")
print("-" * 45)
for method, scores in results.items():
    print(f"  {method:<25} {np.mean(scores) * 100:.2f}")

Loading open-source embedding & cross-encoder models into memory...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading FiQA evaluation parquets...
Evaluating across 50 sample items out of 648 validation queries.


Evaluating RAG Pipelines:   0%|          | 0/50 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


NDCG@10 x100 Results Table (50 sample items)
---------------------------------------------
  BM25                      28.01
  Dense (MiniLM)            35.92
  Hybrid (RRF)              34.79
  Hybrid + Local Rerank     34.01
